# PNNL Bounding-Box Demo

Demonstrates bounding-box queries over the PNNL Richland WA field monitoring site  
for May–June 2023 using two `env-data-mcp` sources:

| Source | Tool | Notes |
|---|---|---|
| NASA POWER | `nasa_power_query` | Centroid of bbox; 0.5° resolution |
| Sentinel-5P TROPOMI | `sentinel5p_bbox_query` | Spatial mean over bbox; one date |

The PNNL bbox is a small field site (~40 m × 30 m).  NASA POWER uses the centroid;  
Sentinel-5P operates at ~3.5 × 5.5 km native resolution so the bbox is well within one pixel.

> **Network required.** Each Sentinel-5P granule is ~150 MB; expect 30–120 s for the S5P cell.

In [ ]:
import json

from env_data_mcp.sources.nasa_power import nasa_power_query
from env_data_mcp.sources.sentinel5p import sentinel5p_bbox_query

# PNNL Richland WA field site (matches tests/conftest.py PNNL_BBOX).
PNNL_BBOX = {
    "min_lat": 46.251407,
    "max_lat": 46.251790,
    "min_lon": -119.728785,
    "max_lon": -119.728369,
}
PNNL_START = "2023-05-01"
PNNL_END = "2023-06-01"

# Centroid for point-based sources.
CENTROID_LAT = (PNNL_BBOX["min_lat"] + PNNL_BBOX["max_lat"]) / 2
CENTROID_LON = (PNNL_BBOX["min_lon"] + PNNL_BBOX["max_lon"]) / 2

print(f"PNNL centroid: ({CENTROID_LAT:.6f}, {CENTROID_LON:.6f})")
print(f"Date range:    {PNNL_START} → {PNNL_END}")

## 1. NASA POWER — daily weather for the PNNL site (May–June 2023)

In [ ]:
nasa_r = nasa_power_query(
    latitude=CENTROID_LAT,
    longitude=CENTROID_LON,
    start_date=PNNL_START,
    end_date=PNNL_END,
)

m = nasa_r["_meta"]
print(f"Success: {m['success']}  |  rows: {m['rows_returned']}  |  latency: {m['latency_s']:.2f}s")
print(f"License: {m['license']}")

if nasa_r["data"]:
    import pandas as pd

    df_nasa = pd.DataFrame(nasa_r["data"])
    print(f"\nColumns: {list(df_nasa.columns)}")
    print(df_nasa[["date", "T2M", "PRECTOTCORR", "RH2M"]].head(10).to_string(index=False))

## 2. Sentinel-5P TROPOMI — CO column mean over bbox (one date)

The bbox is much smaller than a single Sentinel-5P pixel (~3.5 × 5.5 km), so the  
`bbox_query` returns the pixel mean for pixels whose centroids fall within the expanded  
swath covering the site.  We query a single date to stay within the notebook timeout.

In [ ]:
# Single date query — each granule is ~150 MB; expect 30–120 s.
S5P_DATE = "2023-05-15"

s5p_r = sentinel5p_bbox_query(
    min_lat=PNNL_BBOX["min_lat"],
    max_lat=PNNL_BBOX["max_lat"],
    min_lon=PNNL_BBOX["min_lon"],
    max_lon=PNNL_BBOX["max_lon"],
    start_date=S5P_DATE,
    end_date=S5P_DATE,
    product="CO",
)

m = s5p_r["_meta"]
print(f"Success: {m['success']}  |  rows: {m['rows_returned']}  |  latency: {m['latency_s']:.1f}s")
print(f"License: {m['license']}")

if s5p_r["data"]:
    for rec in s5p_r["data"]:
        print(
            f"  date={rec['date']}  CO_mean={rec.get('CO_mean', 'N/A')} mol/m²  "
            f"granule={rec.get('granule_id', 'N/A')}"
        )
else:
    print("  No swath coverage on", S5P_DATE, "— try an adjacent date.")
    if m.get("error"):
        print("  Error:", m["error"])

## 3. Query provenance — `_meta` blocks

Every response carries a `_meta` block with the exact inputs used, latency, licence,  
and variable descriptions.  This is the hook for downstream logging and reproducibility.

In [ ]:
print("NASA POWER _meta:")
print(json.dumps({k: v for k, v in nasa_r["_meta"].items() if k != "variable_info"}, indent=2))
print("\nNASA POWER variable_info keys:", list(nasa_r["_meta"]["variable_info"].keys()))